help me write the code to calculate the range how many SMs under each feeders that have HP, and evs. 
so i want this: 
"across the 20 feeders, there are x-y% SMs that have HPs, and s-r% that have EVs" 


holdout_feeder_id: [291556, 887337, 267697, 129571, 921498, 123538, 615734, 144397, 652352, 942339, 47021, 60068, 641430, 667455, 318813, 732417, 164883, 307389, 657188, 893397]   #291556:  n=42   ev=28.6%   hp=31.0%    avg?=9.70    cover?=100.0%


find information about EVs at /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/data/EV/EV_detection_reduced_60000_SM.csv
EV dataset looks like: 
SM_ID,EV_detected,Type,hp_false_positive_potential,confidence_score,confidence_score_norm,confidence_label,active_phases
sm_100025.parquet,Yes,1ph,No,5,27.586206896551722,Medium,['l2']
sm_100031.parquet,No,None,No,,,,
sm_100042.parquet,No,None,No,,,,
sm_100060.parquet,Yes,1ph,No,65,68.96551724137932,High,['l3']
sm_100066.parquet,Yes,3ph,No,50,58.620689655172406,High,"['l1', 'l2', 'l3']"

find information about HPs at /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/data/HP/label_propagation_predictions.csv
HP dataset looks like: 
,predicted_hp,confidence
l1_100025,False,0.0
l2_100025,False,0.0
l3_100025,False,0.006124284068048742
l2_100031,False,0.0
l3_100031,False,0.0
l1_100042,False,0.020565714971327323
l2_100042,False,0.0
l3_100042,False,0.0
l1_100060,False,0.01092878418397117


find informaiton about which SMs fit to which feeder at /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/data/topology/cleaned_and_corrected/MeterAndCabinet.csv
dataset looks like: 
meter_number,delivery_point_id,cabinet,lv_feeder,has_heat_pump,has_solar_panel,capacity_solar_panel,service_fuse_size
795001,497495,Cabinet.860540,,False,False,,25.0
680196,524547,Cabinet.228973,,False,False,,63.0
31038,524547,Cabinet.228973,,False,False,,63.0

holdout_feeder_id: [291556, 887337, 267697, 129571, 921498, 123538, 615734, 144397, 652352, 942339, 47021, 60068, 641430, 667455, 318813, 732417, 164883, 307389, 657188, 893397]   #291556:  n=42   ev=28.6%   hp=31.0%    avg?=9.70    cover?=100.0%



In [8]:
import pandas as pd
import re

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ev_path = "/Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/data/EV/EV_detection_reduced_60000_SM.csv"
hp_path = "/Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/data/HP/label_propagation_predictions.csv"
meter_cabinet_path = "/Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/data/topology/cleaned_and_corrected/MeterAndCabinet.csv"
topology_path = "/Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/data/topology/cleaned_and_corrected/Topology.csv"

holdout_feeder_id = [
    291556, 887337, 267697, 129571, 921498,
    123538, 615734, 144397, 652352, 942339,
    47021, 60068, 641430, 667455, 318813,
    732417, 164883, 307389, 657188, 893397
]

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------
def extract_id(x):
    """
    Extract the last numeric ID from strings such as:
    - LvFeeder.291556      -> 291556
    - Cabinet.860540      -> 860540
    - sm_100025.parquet   -> 100025
    - l1_100025           -> 100025
    """
    if pd.isna(x):
        return pd.NA

    matches = re.findall(r"\d+", str(x))
    return int(matches[-1]) if matches else pd.NA


# ------------------------------------------------------------
# Load meter-to-cabinet mapping
# ------------------------------------------------------------
meter_cabinet = pd.read_csv(meter_cabinet_path)

meter_cabinet["meter_id"] = meter_cabinet["meter_number"].apply(extract_id)
meter_cabinet["cabinet_id"] = meter_cabinet["cabinet"].apply(extract_id)

meter_cabinet = meter_cabinet.dropna(subset=["meter_id", "cabinet_id"]).copy()

meter_cabinet["meter_id"] = meter_cabinet["meter_id"].astype(int)
meter_cabinet["cabinet_id"] = meter_cabinet["cabinet_id"].astype(int)

meter_cabinet = meter_cabinet.drop_duplicates(
    subset=["meter_id", "cabinet_id"]
)

# ------------------------------------------------------------
# Load topology and create cabinet-to-feeder mapping
# ------------------------------------------------------------
topology = pd.read_csv(topology_path)

topology["lv_feeder_id"] = topology["lv_feeder"].apply(extract_id)
topology["node1_cabinet_id"] = topology["node1"].apply(extract_id)
topology["node2_cabinet_id"] = topology["node2"].apply(extract_id)

# Cabinet can appear as node1 or node2, so stack both mappings
cabinet_feeder_node1 = topology[["node1_cabinet_id", "lv_feeder_id"]].rename(
    columns={"node1_cabinet_id": "cabinet_id"}
)

cabinet_feeder_node2 = topology[["node2_cabinet_id", "lv_feeder_id"]].rename(
    columns={"node2_cabinet_id": "cabinet_id"}
)

cabinet_feeder = pd.concat(
    [cabinet_feeder_node1, cabinet_feeder_node2],
    ignore_index=True
)

cabinet_feeder = cabinet_feeder.dropna(subset=["cabinet_id", "lv_feeder_id"]).copy()

cabinet_feeder["cabinet_id"] = cabinet_feeder["cabinet_id"].astype(int)
cabinet_feeder["lv_feeder_id"] = cabinet_feeder["lv_feeder_id"].astype(int)

cabinet_feeder = cabinet_feeder.drop_duplicates(
    subset=["cabinet_id", "lv_feeder_id"]
)

# ------------------------------------------------------------
# Map meters to feeders via cabinets
# ------------------------------------------------------------
meter_feeder = (
    meter_cabinet[["meter_id", "cabinet_id"]]
    .merge(cabinet_feeder, on="cabinet_id", how="left")
)

# Keep only holdout feeders
holdout_meter_feeder = meter_feeder[
    meter_feeder["lv_feeder_id"].isin(holdout_feeder_id)
].copy()

holdout_meter_feeder = holdout_meter_feeder.drop_duplicates(
    subset=["meter_id", "lv_feeder_id"]
)

print("Matched rows:", len(holdout_meter_feeder))
print("Matched feeders:", sorted(holdout_meter_feeder["lv_feeder_id"].dropna().unique()))
print("Number of matched feeders:", holdout_meter_feeder["lv_feeder_id"].nunique())

missing_feeders = sorted(set(holdout_feeder_id) - set(holdout_meter_feeder["lv_feeder_id"].dropna().unique()))
print("Missing holdout feeders:", missing_feeders)

if holdout_meter_feeder.empty:
    raise ValueError(
        "No holdout feeders matched after mapping cabinet -> lv_feeder. "
        "Check whether the holdout IDs are lv_feeder IDs and whether the cabinet IDs match between files."
    )

# ------------------------------------------------------------
# Load EV labels
# ------------------------------------------------------------
ev = pd.read_csv(ev_path)

ev["meter_id"] = ev["SM_ID"].apply(extract_id)
ev["has_ev"] = ev["EV_detected"].astype(str).str.lower().eq("yes")

ev_meter = (
    ev.groupby("meter_id", as_index=False)["has_ev"]
    .max()
)

# ------------------------------------------------------------
# Load HP labels
# ------------------------------------------------------------
hp = pd.read_csv(hp_path)

hp_id_col = hp.columns[0]

hp["meter_id"] = hp[hp_id_col].apply(extract_id)
hp["has_hp"] = hp["predicted_hp"].astype(str).str.lower().eq("true")

# HP file is phase-level, so aggregate to SM-level:
# an SM has HP if any phase is predicted as HP.
hp_meter = (
    hp.groupby("meter_id", as_index=False)["has_hp"]
    .max()
)

# ------------------------------------------------------------
# Join EV and HP labels onto holdout meter-feeder mapping
# ------------------------------------------------------------
df = (
    holdout_meter_feeder[["meter_id", "lv_feeder_id"]]
    .merge(ev_meter, on="meter_id", how="left")
    .merge(hp_meter, on="meter_id", how="left")
)

# Missing labels are treated as False.
df["has_ev"] = df["has_ev"].fillna(False)
df["has_hp"] = df["has_hp"].fillna(False)

# ------------------------------------------------------------
# Feeder-level summary
# ------------------------------------------------------------
feeder_summary = (
    df.groupby("lv_feeder_id")
    .agg(
        n_sms=("meter_id", "nunique"),
        n_ev=("has_ev", "sum"),
        n_hp=("has_hp", "sum"),
        ev_pct=("has_ev", lambda x: 100 * x.mean()),
        hp_pct=("has_hp", lambda x: 100 * x.mean()),
    )
    .reset_index()
    .sort_values("lv_feeder_id")
)

feeder_summary["ev_pct"] = feeder_summary["ev_pct"].round(1)
feeder_summary["hp_pct"] = feeder_summary["hp_pct"].round(1)

print(feeder_summary)

# ------------------------------------------------------------
# Range across holdout feeders
# ------------------------------------------------------------
hp_min = feeder_summary["hp_pct"].min()
hp_max = feeder_summary["hp_pct"].max()

ev_min = feeder_summary["ev_pct"].min()
ev_max = feeder_summary["ev_pct"].max()

print(
    f"Across the {feeder_summary['lv_feeder_id'].nunique()} feeders, "
    f"{hp_min:.1f}-{hp_max:.1f}% of SMs have HPs, "
    f"and {ev_min:.1f}-{ev_max:.1f}% have EVs. "
    f"Average across feeders: {feeder_summary['hp_pct'].mean():.1f}% HP, {feeder_summary['ev_pct'].mean():.1f}% EV."
)

Matched rows: 707
Matched feeders: [np.int64(47021), np.int64(60068), np.int64(123538), np.int64(129571), np.int64(144397), np.int64(164883), np.int64(267697), np.int64(291556), np.int64(307389), np.int64(318813), np.int64(615734), np.int64(641430), np.int64(652352), np.int64(657188), np.int64(667455), np.int64(732417), np.int64(887337), np.int64(893397), np.int64(921498), np.int64(942339)]
Number of matched feeders: 20
Missing holdout feeders: []
    lv_feeder_id  n_sms n_ev n_hp  ev_pct  hp_pct
0          47021     36    7   30    19.4    83.3
1          60068     32    8    2    25.0     6.2
2         123538     20   17    6    85.0    30.0
3         129571     62   22   43    35.5    69.4
4         144397     79    1    1     1.3     1.3
5         164883     36    8   16    22.2    44.4
6         267697      9    3    5    33.3    55.6
7         291556     42   12   13    28.6    31.0
8         307389     78    1    4     1.3     5.1
9         318813      8    4    4    50.0    50.